# Stage 11 — Stock Tracker: Current Inventory Position
**Dashboard page:** Inventory Analysis (Inventory tab) · Inventory Status (standalone)
**Tabs:** All · At-Risk · Excess · Storage Locations · Coverage Histogram

**Source priority:** SAP current_stock.parquet (authoritative) → movement reconstruction (fallback)
**Excluded locations:** Damage · GR Unavailable · 0020 · 0030

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
st   = load("stock_tracker.parquet")
curr = load("current_stock.parquet") if (INTERIM/"current_stock.parquet").exists() else pd.DataFrame()

print(f"Stock tracker  : {len(st):,} SKUs | cols: {st.columns.tolist()}")
if len(curr): print(f"Current stock  : {len(curr):,} rows | cols: {curr.columns.tolist()}")
print()
print("Stock status breakdown:")
print(st["stock_status"].value_counts().to_string())


## Stock Status Distribution (All Tab)

In [ ]:
order  = ["stockout","critical","low","ok","excess"]
sc = st["stock_status"].value_counts().reindex([x for x in order if x in st["stock_status"].values],fill_value=0)
colors = [STATUS_COLORS.get(s,"#94A3B8") for s in sc.index]
fig,axes = plt.subplots(1,2,figsize=(13,4))
sc.plot(kind="bar",ax=axes[0],color=colors,edgecolor="white")
axes[0].set_title("Stock Status Distribution
(stockout≤0, critical<1m, low<3m, ok<6m, excess≥6m)")
axes[0].tick_params(axis="x",rotation=0)
for bar,val in zip(axes[0].patches,sc.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+3,str(val),ha="center",fontsize=9)
axes[1].pie(sc.values,labels=sc.index,colors=colors,autopct="%1.0f%%",startangle=90)
axes[1].set_title("Status Share"); plt.tight_layout(); plt.show()
for s,cnt in sc.items(): print(f"  {s:12s}: {cnt:5,} ({cnt/len(st)*100:.1f}%)")


## Coverage Histogram

In [ ]:
cov = st["coverage_months"].replace(999,np.nan).clip(upper=24)
fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].hist(cov.dropna(),bins=60,color=PALETTE[0],edgecolor="white",alpha=0.8)
axes[0].axvline(3,color="#EF4444",ls="--",lw=1.8,label="Lead time = 3 months")
axes[0].axvline(6,color="#F59E0B",ls="--",lw=1.5,label="Excess threshold = 6 months")
axes[0].set_title("Coverage Months Distribution (active SKUs, clipped at 24m)
"
                  "999=non-moving, excluded")
axes[0].set_xlabel("Months of stock"); axes[0].legend(fontsize=8)

axes[1].scatter(st["avg_monthly_demand"].clip(upper=st["avg_monthly_demand"].quantile(0.97)),
                st["stock_on_hand"].clip(upper=st["stock_on_hand"].quantile(0.97)),
                alpha=0.25,s=10,color=PALETTE[0])
axes[1].set_title("Stock on Hand vs Avg Monthly Demand")
axes[1].set_xlabel("Avg monthly demand"); axes[1].set_ylabel("Stock on hand")
plt.tight_layout(); plt.show()


## At-Risk Tab (coverage < 1 month or stockout)

In [ ]:
at_risk = st[st["stock_status"].isin(["stockout","critical"])].copy()
at_risk_low = st[st["stock_status"]=="low"].copy()
print(f"Stockout + Critical : {len(at_risk):,} SKUs")
print(f"Low coverage        : {len(at_risk_low):,} SKUs")
print()
top_risk = at_risk.nsmallest(20,"coverage_months")[
    ["material_9","description","stock_on_hand","coverage_months","stock_status","forecast_lt"]]
print("Top 20 most at-risk SKUs:")
print(top_risk.to_string(index=False))

if len(at_risk)>0:
    fig,ax = plt.subplots(figsize=(11,4))
    at_risk["coverage_months"].hist(bins=40,ax=ax,color=PALETTE[1],edgecolor="white",alpha=0.8)
    ax.set_title(f"At-Risk SKUs: Coverage Distribution ({len(at_risk):,} SKUs)")
    ax.set_xlabel("Coverage months"); plt.tight_layout(); plt.show()


## Excess Tab (coverage > 6 months)

In [ ]:
excess = st[st["stock_status"]=="excess"].copy()
print(f"Excess SKUs: {len(excess):,}")
excess_val = (excess["stock_on_hand"] * excess["unit_value_lkr"]).sum() if "unit_value_lkr" in excess.columns else 0
print(f"Excess stock value: {fmt_lkr(excess_val)}")
top_excess = excess.nlargest(20,"coverage_months")[
    ["material_9","description","stock_on_hand","coverage_months","avg_monthly_demand"]]
print("
Top 20 excess SKUs by coverage:")
print(top_excess.to_string(index=False))

if len(excess)>0:
    fig,ax = plt.subplots(figsize=(11,4))
    excess["coverage_months"].clip(upper=24).hist(bins=40,ax=ax,color=PALETTE[4],edgecolor="white",alpha=0.8)
    ax.axvline(6,color=PALETTE[1],ls="--",lw=1.5,label="Excess threshold=6m")
    ax.set_title(f"Excess SKUs: Coverage Distribution ({len(excess):,} SKUs, clipped at 24m)")
    ax.set_xlabel("Coverage months"); ax.legend(); plt.tight_layout(); plt.show()


## Storage Location Breakdown

In [ ]:
if len(curr)>0:
    loc_col = next((c for c in ["StorageLoc","storage_location","sloc","Sloc"] if c in curr.columns),None)
    qty_col = next((c for c in ["Unrestricted","qty","quantity","Stock"] if c in curr.columns),None)
    if loc_col and qty_col:
        by_loc = curr.groupby(loc_col)[qty_col].sum().sort_values(ascending=False)
        fig,ax = plt.subplots(figsize=(11,4))
        by_loc.head(20).plot(kind="bar",ax=ax,color=PALETTE[0],edgecolor="white")
        ax.set_title("Stock Qty by SAP Storage Location (Top 20)
"
                     "Damage/GR Unavailable/0020/0030 excluded from active tracker")
        ax.set_ylabel("Total qty"); ax.tick_params(axis="x",rotation=45)
        plt.tight_layout(); plt.show()
        print(by_loc.to_string())
else:
    print("current_stock.parquet not found or empty.")
